# Week 10: BoVW → MLP in PyTorch

**Topics:** PyTorch crash course (numpy ↔ torch, autograd), reusable BoVW
feature extractor, training a multi-layer perceptron with the
forward / backward / step ritual, evaluating with accuracy / confusion
matrix / precision / recall / F1, and two head-to-head comparisons
(linear vs MLP, sigmoid vs ReLU).

Last week the classifier was a black box on top of BoVW. Today we open
the box and **learn** the classifier ourselves.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import cv2

%matplotlib inline

### Environment setup

Works **locally** and on **Google Colab**. STL-10 (~2.5 GB) is downloaded
into a local cache the first time the notebook runs.

In [ ]:
import os

IN_COLAB = "google.colab" in str(get_ipython()) if hasattr(__builtins__, "__IPYTHON__") else False

if IN_COLAB:
    DATA_DIR = "/content/stl10"
else:
    DATA_DIR = os.path.expanduser("~/.cache/stl10")

os.makedirs(DATA_DIR, exist_ok=True)
print(f"Running on: {'Google Colab' if IN_COLAB else 'Local'}")
print(f"STL-10 cache: {DATA_DIR}")

### Display helpers

In [ ]:
def show_image(img, title=None, scale=4):
    """Display a single image. Grayscale arrays auto-use gray colormap."""
    fig, ax = plt.subplots(figsize=(scale, scale))
    if img.ndim == 2:
        ax.imshow(img, cmap="gray", vmin=0, vmax=255)
    else:
        ax.imshow(img)
    if title:
        ax.set_title(title)
    ax.axis("off")
    plt.tight_layout()
    plt.show()


def show_bovw_histogram(hist, title=None, scale=(8, 3)):
    """Plot a single BoVW histogram (length K = number of visual words)."""
    fig, ax = plt.subplots(figsize=scale)
    K = len(hist)
    ax.bar(np.arange(K), hist, width=1.0)
    ax.set_xlim([-0.5, K - 0.5])
    ax.set_xlabel("Visual word index")
    ax.set_ylabel("Frequency")
    if title:
        ax.set_title(title)
    plt.tight_layout()
    plt.show()


def show_confusion_matrix(cm, class_names, title=None, scale=5):
    """Display a confusion matrix as an annotated heatmap."""
    fig, ax = plt.subplots(figsize=(scale, scale))
    im = ax.imshow(cm, cmap="Blues")
    plt.colorbar(im, ax=ax, fraction=0.046)
    n = len(class_names)
    ax.set_xticks(np.arange(n))
    ax.set_yticks(np.arange(n))
    ax.set_xticklabels(class_names, rotation=45, ha="right")
    ax.set_yticklabels(class_names)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    thresh = cm.max() / 2 if cm.max() > 0 else 0
    for i in range(n):
        for j in range(n):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > thresh else "black",
                    fontsize=10)
    if title:
        ax.set_title(title)
    plt.tight_layout()
    plt.show()

---

## 0. PyTorch Crash Course

PyTorch is a numerical library, just like NumPy — same idea of N-dim arrays
and elementwise ops. The two new things on top are:

1. Tensors live on a **device** (CPU or GPU) and carry a **dtype**.
2. Tensors can track **gradients** — PyTorch computes derivatives for you.

That's it. Everything else (`nn.Linear`, `nn.ReLU`, `optim.Adam`, ...) is
convenience built on top of these two ideas.

In [ ]:
import torch
import torch.nn as nn

### NumPy ↔ Torch

Convert in either direction. Note the dtype — PyTorch is stricter than
NumPy about float32 vs float64.

In [ ]:
a_np = np.array([[1.0, 2.0, 3.0],
                 [4.0, 5.0, 6.0]])
print(f"a_np.shape = {a_np.shape}, dtype = {a_np.dtype}")

# numpy -> torch
a_t = torch.from_numpy(a_np)
print(f"a_t.shape  = {tuple(a_t.shape)}, dtype = {a_t.dtype}")

# torch -> numpy
back = a_t.numpy()
print(f"back.shape = {back.shape}, dtype = {back.dtype}")

In [ ]:
# Cast dtype with .float() / .long() / .to(torch.float32)
x = torch.from_numpy(a_np).float()        # float32
y = torch.tensor([0, 1], dtype=torch.long)  # int64
print(f"x.dtype = {x.dtype}")
print(f"y.dtype = {y.dtype}")

### Basic ops feel like NumPy

Shape, broadcasting, and matmul all work the same way. The matmul operator
is `@` (this is also valid Python NumPy syntax).

In [ ]:
A = torch.tensor([[1.0, 2.0],
                  [3.0, 4.0]])
b = torch.tensor([10.0, 20.0])

print(f"A + b = \n{A + b}")          # broadcasting (b added to each row)
print(f"A @ b = {A @ b}")              # matrix-vector product
print(f"A.sum(dim=0) = {A.sum(dim=0)}")  # column sums

### Autograd — the one thing NumPy can't do

Mark a tensor with `requires_grad=True`. PyTorch records every op into a
graph. When you call `.backward()` on a scalar, gradients flow back and
land in `.grad` of the leaves.

If $y = x^2$, then $\dfrac{dy}{dx} = 2x$. At $x = 3$, the gradient is $6$.

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2
y.backward()
print(f"x.grad = {x.grad}")    # should print 6.0

### `nn.Linear` — a learnable affine layer

`nn.Linear(in_features, out_features)` creates an affine map
$\mathbf{y} = W\mathbf{x} + \mathbf{b}$ with **trainable** $W$ and $\mathbf{b}$.
It's a tensor with autograd already wired up — you don't manage gradients
by hand.

In [ ]:
layer = nn.Linear(in_features=4, out_features=2)

x = torch.randn(1, 4)        # one sample, 4 features
out = layer(x)

print(f"x.shape          = {tuple(x.shape)}")
print(f"out.shape        = {tuple(out.shape)}")
print(f"weight.shape     = {tuple(layer.weight.shape)}   # (out, in)")
print(f"bias.shape       = {tuple(layer.bias.shape)}")

---

## 1. The BoVW Extractor — Image → 200-dim Vector

We treat the BoVW pipeline as a **black box** this week. Last week we
built it from scratch (SIFT → k-means → histogram); today we just *use*
it. The function `extract_bovw(img)` turns any grayscale image into a
200-dim vector, and that vector is what we'll feed into the MLP.

In [ ]:
from torchvision.datasets import STL10

train_ds = STL10(root=DATA_DIR, split="train", download=True)
test_ds = STL10(root=DATA_DIR, split="test", download=True)

print(f"len(train_ds) = {len(train_ds)}")
print(f"len(test_ds)  = {len(test_ds)}")

Same 5-class subset as week 9: `airplane`, `bird`, `car`, `cat`, `ship`,
remapped to labels `0..4`.

In [ ]:
SELECTED = [0, 1, 2, 3, 8]
PER_CLASS_TRAIN = 200
PER_CLASS_TEST = 30

class_names = []
for c in SELECTED:
    class_names.append(train_ds.classes[c])


def select_subset(ds, n_per_class):
    """Keep `n_per_class` images from each SELECTED class; remap labels to 0..N-1."""
    new_label = {}
    for new_idx, old_label in enumerate(SELECTED):
        new_label[old_label] = new_idx
    counts = {}
    for c in SELECTED:
        counts[c] = 0
    items = []
    for i in range(len(ds)):
        img, label = ds[i]
        if label not in new_label:
            continue
        if counts[label] >= n_per_class:
            continue
        gray = np.array(img.convert("L"))
        items.append((gray, new_label[label]))
        counts[label] += 1
    return items

train_items = select_subset(train_ds, PER_CLASS_TRAIN)
test_items = select_subset(test_ds, PER_CLASS_TEST)
print(f"class_names      = {class_names}")
print(f"len(train_items) = {len(train_items)}")
print(f"len(test_items)  = {len(test_items)}")

In [ ]:
from sklearn.cluster import MiniBatchKMeans

K_VOCAB = 200
N_DESC_CAP = 100   # keep top-N keypoints per image (by response strength)

sift = cv2.SIFT_create()

# 1) Extract SIFT descriptors for every training image
train_desc = []
for gray, lab in train_items:
    kp, des = sift.detectAndCompute(gray, None)
    if des is None or len(des) == 0:
        train_desc.append(np.zeros((0, 128), dtype=np.float32))
        continue
    if len(des) > N_DESC_CAP:
        responses = []
        for k in kp:
            responses.append(k.response)
        responses = np.array(responses)
        top_idx = np.argpartition(-responses, N_DESC_CAP)[:N_DESC_CAP]
        des = des[top_idx]
    train_desc.append(des.astype(np.float32))

# 2) Stack into one big matrix and fit k-means
non_empty = []
for d in train_desc:
    if len(d) > 0:
        non_empty.append(d)
all_descriptors = np.vstack(non_empty)
print(f"all_descriptors.shape = {all_descriptors.shape}")

kmeans = MiniBatchKMeans(n_clusters=K_VOCAB, batch_size=1024, n_init=3, random_state=0)
kmeans.fit(all_descriptors)
print(f"kmeans.cluster_centers_.shape = {kmeans.cluster_centers_.shape}")

from sklearn.cluster import MiniBatchKMeans

K_VOCAB = 200
sift = cv2.SIFT_create()

# Fit the visual vocabulary on training SIFT descriptors (black-boxed; same pipeline as week 9)
all_desc = []
for gray, _ in train_items:
    kp, des = sift.detectAndCompute(gray, None)
    if des is None or len(des) == 0:
        continue
    if len(des) > 100:
        responses = []
        for k in kp:
            responses.append(k.response)
        idx = np.argpartition(-np.array(responses), 100)[:100]
        des = des[idx]
    all_desc.append(des.astype(np.float32))

kmeans = MiniBatchKMeans(n_clusters=K_VOCAB, batch_size=1024, n_init=3, random_state=0)
kmeans.fit(np.vstack(all_desc))

In [ ]:
def extract_bovw(img_gray, sift, kmeans, K):
    """Extract a BoVW histogram from one grayscale image.

    img_gray: (H, W) uint8 array
    Returns:  (K,) float32 array, L1-normalized so it sums to 1.
    """
    kp, des = sift.detectAndCompute(img_gray, None)
    if des is None or len(des) == 0:
        return np.zeros(K, dtype=np.float32)
    words = kmeans.predict(des)
    hist = np.bincount(words, minlength=K).astype(np.float32)
    return hist / hist.sum()

### How does the BoVW look for a few images?

Different classes should produce noticeably different histograms — that
**is** the signal the classifier will pick up on.

In [ ]:
# One image from each of four classes
demo_classes = [0, 1, 3, 4]   # airplane, bird, cat, ship

for cls in demo_classes:
    for gray, lab in train_items:
        if lab == cls:
            show_image(gray, title=f"{class_names[cls]}")
            hist = extract_bovw(gray, sift, kmeans, K_VOCAB)
            show_bovw_histogram(hist, title=f"BoVW of {class_names[cls]}")
            break

### Apply `extract_bovw` to every image → `X_train`, `X_test`

From here on, an image is just a 200-dim vector.

In [ ]:
X_train = np.zeros((len(train_items), K_VOCAB), dtype=np.float32)
y_train = np.zeros(len(train_items), dtype=np.int64)
for i in range(len(train_items)):
    gray, lab = train_items[i]
    X_train[i] = extract_bovw(gray, sift, kmeans, K_VOCAB)
    y_train[i] = lab

X_test = np.zeros((len(test_items), K_VOCAB), dtype=np.float32)
y_test = np.zeros(len(test_items), dtype=np.int64)
for i in range(len(test_items)):
    gray, lab = test_items[i]
    X_test[i] = extract_bovw(gray, sift, kmeans, K_VOCAB)
    y_test[i] = lab

print(f"X_train.shape = {X_train.shape}, dtype = {X_train.dtype}")
print(f"y_train.shape = {y_train.shape}, dtype = {y_train.dtype}")
print(f"X_test.shape  = {X_test.shape}")
print(f"y_test.shape  = {y_test.shape}")

Convert once to torch tensors. **Features stay float32, labels become long.**
(`nn.CrossEntropyLoss` expects integer labels of dtype `long`.)

In [ ]:
X_train_t = torch.from_numpy(X_train)
y_train_t = torch.from_numpy(y_train)
X_test_t = torch.from_numpy(X_test)
y_test_t = torch.from_numpy(y_test)

print(f"X_train_t.dtype = {X_train_t.dtype}")
print(f"y_train_t.dtype = {y_train_t.dtype}")

---

## 2. Define an MLP

An MLP is a stack of affine layers separated by a nonlinearity. Without
the nonlinearity, two stacked `nn.Linear`s collapse into one (recall
*Linearity Collapse* from the lecture).

Architecture: **`200 → 32 → 5`** with a single hidden layer and ReLU.

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(200, 32)
        self.fc2 = nn.Linear(32, 5)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

Inspect the model and run **one untrained forward pass** to verify shapes.

In [ ]:
model = MLP()
print(model)

# Forward 8 samples through the untrained network
out = model(X_train_t[:8])
print(f"out.shape = {tuple(out.shape)}   # (batch, num_classes)")

# Total trainable parameters
n_params = 0
for p in model.parameters():
    n_params += p.numel()
print(f"trainable params = {n_params}")

---

## 3. Train the MLP — forward / backward / step

Three ingredients:
- **Loss** — `nn.CrossEntropyLoss` (multi-class softmax + cross-entropy in one).
- **Optimizer** — `Adam`, a smarter cousin of vanilla GD.
- **Loop** — forward, compute loss, backward, step, zero gradients.

In [ ]:
from torch import optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-2)

### One step at a time

The four lines below are the entire "learning" mechanism. Every training
loop in deep learning is a `for` over these four lines.

In [ ]:
logits = model(X_train_t)              # 1) forward
loss = criterion(logits, y_train_t)     # 2) compare to targets
print(f"loss before update = {loss.item():.4f}")

loss.backward()                         # 3) gradients flow back to every weight
optimizer.step()                        # 4) weights move one step downhill
optimizer.zero_grad()                   #    reset gradients for the next iteration

logits = model(X_train_t)
loss = criterion(logits, y_train_t)
print(f"loss after  update = {loss.item():.4f}")

### Now in a loop

In [ ]:
losses = []
for epoch in range(100):
    logits = model(X_train_t)
    loss = criterion(logits, y_train_t)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if (epoch + 1) % 10 == 0:
        print(f"epoch {epoch + 1:3d}   loss {loss.item():.4f}")

In [ ]:
plt.figure(figsize=(6, 3))
plt.plot(losses)
plt.xlabel("epoch")
plt.ylabel("training loss")
plt.title("MLP training loss")
plt.tight_layout()
plt.show()

---

## 4. Evaluation — Accuracy, Confusion Matrix, P/R/F1

On the test set, freeze the model with `torch.no_grad()` (no gradient
graph needed for inference) and read off the predicted class with
`argmax`.

In [ ]:
with torch.no_grad():
    logits = model(X_test_t)
    y_pred = logits.argmax(dim=1).numpy()

accuracy = (y_pred == y_test).mean()
print(f"test accuracy = {accuracy:.3f}")

### Confusion matrix

`cm[i, j]` = number of samples whose **true** class is `i` and whose
**predicted** class is `j`. The diagonal is correct predictions; off-
diagonal entries are class-pair confusions.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

cm = confusion_matrix(y_test, y_pred)
print(f"cm.shape = {cm.shape}")
show_confusion_matrix(cm, class_names, title="MLP — test confusion matrix")

### Precision, Recall, F1 for one class

Pick one class as the **positive** class — say, `ship`. Treat the other
four as one big negative class. Then:

$$
\text{Precision} = \frac{\text{TP}}{\text{TP} + \text{FP}}, \quad
\text{Recall} = \frac{\text{TP}}{\text{TP} + \text{FN}}, \quad
F_1 = \frac{2 \cdot P \cdot R}{P + R}
$$

Read the counts straight off the confusion matrix:
- **TP** = `cm[ship, ship]`
- **FP** = column `ship` minus the diagonal (other classes predicted as ship)
- **FN** = row `ship` minus the diagonal (ship predicted as something else)

In [ ]:
c = class_names.index("ship")
TP = cm[c, c]
FP = cm[:, c].sum() - TP
FN = cm[c, :].sum() - TP

precision = TP / (TP + FP)
recall    = TP / (TP + FN)
f1        = 2 * precision * recall / (precision + recall)

print(f"ship   TP={TP}  FP={FP}  FN={FN}")
print(f"       precision = {precision:.3f}")
print(f"       recall    = {recall:.3f}")
print(f"       F1        = {f1:.3f}")

Sanity check against sklearn's `classification_report` — the `ship` row
should match the numbers we just hand-computed.

In [ ]:
print(classification_report(y_test, y_pred, target_names=class_names, digits=3))

---

## 5. Two Quick Comparisons

Let's pressure-test two claims from the lecture on real data:

- **5A** — "a hidden layer + nonlinearity beats a linear classifier"
- **5B** — "ReLU trains better than sigmoid in deeper nets"

We're going to retrain a few models, so it pays to package the loop into
a tiny helper. Same four lines you saw above — just wrapped.

In [ ]:
def train_model(model, X, y, lr, epochs):
    """Train with Adam + CrossEntropyLoss; return loss history."""
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    losses = []
    for epoch in range(epochs):
        logits = model(X)
        loss = criterion(logits, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return losses


def test_accuracy(model, X, y):
    with torch.no_grad():
        pred = model(X).argmax(dim=1).numpy()
    return (pred == y.numpy()).mean()

### 5A. Linear classifier vs MLP

Strip every hidden layer and ReLU — just a single `nn.Linear`. This is
logistic regression for $C$ classes (multinomial). Train it the same way.

In [ ]:
class LinearOnly(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(200, 5)

    def forward(self, x):
        return self.fc(x)


torch.manual_seed(0)
linear_model = LinearOnly()
linear_losses = train_model(linear_model, X_train_t, y_train_t, lr=1e-2, epochs=100)
linear_acc = test_accuracy(linear_model, X_test_t, y_test_t)

torch.manual_seed(0)
mlp_model = MLP()
mlp_losses = train_model(mlp_model, X_train_t, y_train_t, lr=1e-2, epochs=100)
mlp_acc = test_accuracy(mlp_model, X_test_t, y_test_t)

print(f"Linear only test acc = {linear_acc:.3f}")
print(f"MLP        test acc = {mlp_acc:.3f}")

In [ ]:
plt.figure(figsize=(6, 3))
plt.plot(linear_losses, label=f"Linear (acc {linear_acc:.2f})")
plt.plot(mlp_losses,    label=f"MLP    (acc {mlp_acc:.2f})")
plt.xlabel("epoch")
plt.ylabel("training loss")
plt.title("Linear classifier vs MLP")
plt.legend()
plt.tight_layout()
plt.show()

### 5B. Sigmoid vs ReLU

Same MLP architecture, only the activation differs. Even with one hidden
layer, sigmoid's saturating output pinches the gradient — the loss curve
should descend visibly slower than ReLU's.

In [ ]:
class MLP_Sigmoid(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(200, 32)
        self.fc2 = nn.Linear(32, 5)
        self.act = nn.Sigmoid()

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.fc2(x)
        return x


torch.manual_seed(0)
sig_model = MLP_Sigmoid()
sig_losses = train_model(sig_model, X_train_t, y_train_t, lr=1e-2, epochs=100)
sig_acc = test_accuracy(sig_model, X_test_t, y_test_t)

print(f"Sigmoid MLP test acc = {sig_acc:.3f}")
print(f"ReLU    MLP test acc = {mlp_acc:.3f}")

In [ ]:
plt.figure(figsize=(6, 3))
plt.plot(mlp_losses, label=f"ReLU    (acc {mlp_acc:.2f})")
plt.plot(sig_losses, label=f"Sigmoid (acc {sig_acc:.2f})")
plt.xlabel("epoch")
plt.ylabel("training loss")
plt.title("ReLU vs Sigmoid in the same MLP")
plt.legend()
plt.tight_layout()
plt.show()

---

## 6. Exercise — Learning Rate Sweep

From the lecture: *learning rate too small → painfully slow; too large
→ the loss bounces or diverges.* Let's see this on real data.

Train the **ReLU MLP** five times, once for each learning rate in
`[1e-4, 1e-3, 1e-2, 1e-1, 1.0]`. Plot all five loss curves on the same
axes with a legend.

**Hint:** Reset the model with `MLP()` *before* each run — otherwise the
weights from the previous run carry over and the comparison is unfair.

In [ ]:
lrs = [1e-4, 1e-3, 1e-2, 1e-1, 1.0]

# YOUR CODE HERE
# For each lr in `lrs`:
#   - create a fresh model with MLP() and torch.manual_seed(0)
#   - call train_model(...) for 100 epochs and collect the loss list
# Plot all 5 loss curves on one figure with a legend.


# Which learning rate converges fastest? Which one diverges?
# What does that tell you about the "step size" intuition from the lecture?
# (You may answer in Korean.)
#
